In [ ]:
import os
import requests

# Read the key from an environment variable instead of hardcoding it
API_KEY = os.environ.get("OPENAQ_API_KEY", "bf008b449b0290c59067e4c0072c9dacb6d6fe10c63ebf174d3a9736c2c334a8")
BASE_URL = "https://api.openaq.org/v3"
HEADERS = {"X-API-Key": API_KEY}

# Kathmandu Valley bounding box: min_lon, min_lat, max_lon, max_lat
KATHMANDU_BBOX = "85.20,27.55,85.60,27.85"
PM25_PARAMETER_ID = 2


def get_valley_stations():
    """Return all stations inside the Kathmandu Valley that measure PM2.5."""
    params = {
        "bbox": KATHMANDU_BBOX,
        "parameters_id": PM25_PARAMETER_ID,
        "limit": 100,
    }
    response = requests.get(f"{BASE_URL}/locations", headers=HEADERS, params=params, timeout=30)
    response.raise_for_status()
    return response.json().get("results", [])


def get_latest_pm25(location):
    """Return (value, timestamp) of the latest PM2.5 reading for a location."""
    # Map sensor IDs to parameter names for this location
    pm25_sensor_ids = {
        sensor["id"]
        for sensor in location.get("sensors", [])
        if sensor.get("parameter", {}).get("id") == PM25_PARAMETER_ID
    }

    response = requests.get(
        f"{BASE_URL}/locations/{location['id']}/latest",
        headers=HEADERS,
        timeout=30,
    )
    response.raise_for_status()

    for reading in response.json().get("results", []):
        if reading.get("sensorsId") in pm25_sensor_ids:
            timestamp = reading.get("datetime", {}).get("local")
            return reading.get("value"), timestamp
    return None, None


def main():
    print("Fetching Kathmandu Valley PM2.5 stations from OpenAQ v3...\n")
    try:
        stations = get_valley_stations()
    except requests.HTTPError as err:
        print(f"Request failed: {err}\n{err.response.text}")
        return

    if not stations:
        print("No stations found in the specified area.")
        return

    print(f"Stations found: {len(stations)}\n")
    for station in stations:
        try:
            value, timestamp = get_latest_pm25(station)
        except requests.HTTPError as err:
            value, timestamp = None, f"error {err.response.status_code}"

        coords = station.get("coordinates", {})
        print(f"Station:     {station.get('name')}")
        print(f"ID:          {station.get('id')}")
        print(f"Coordinates: {coords.get('latitude')}, {coords.get('longitude')}")
        print(f"Latest PM2.5: {value} µg/m³ (at {timestamp})")
        print()


if __name__ == "__main__":
    main()

Fetching Kathmandu Valley PM2.5 stations from OpenAQ v3...

Stations found: 62

Station:     Embassy Kathmandu
ID:          3459
Coordinates: 27.738703, 85.336206
Latest PM2.5: 21.0 µg/m³ (at 2026-06-18T05:00:00+05:45)

Station:     Phora Durbar Kathman
ID:          3460
Coordinates: 27.712464, 85.315703
Latest PM2.5: 71.0 µg/m³ (at 2025-03-24T20:00:00+05:45)

Station:     Dhathutole, Handigaun
ID:          1236017
Coordinates: 27.727502, 85.330135
Latest PM2.5: 47.7 µg/m³ (at 2026-06-01T09:45:00+05:45)

Station:     Dabali, Handigaun
ID:          2848864
Coordinates: 27.68189114, 85.28707804
Latest PM2.5: 42.5 µg/m³ (at 2026-10-19T08:45:00+05:45)

Station:     Kathmandu Institute of Applied Sciences (KIAS)
ID:          4121681
Coordinates: 27.6716935, 85.30167
Latest PM2.5: 43.4 µg/m³ (at 2026-11-20T16:45:00+05:45)

Station:     Gaushala Chowk (SC-01) - GD Labs
ID:          5506835
Coordinates: 27.707763, 85.343189
Latest PM2.5: 46.8 µg/m³ (at 2026-11-20T16:45:00+05:45)

Station:     

KeyboardInterrupt: 